In [5]:
%env DATA_PATH=../../../data
from db import *
from api.routers.rolls import get_roll
from sqlalchemy import select
from sqlalchemy.orm import selectinload
from lib.paths import resolve_path
from lib.fit import load_fit_file, get_gps_data, estimate_fit_timestamp, get_fit_graph_data, get_sensor_data, get_camera_starts, get_camera_ends
from lib.racebox import get_racebox_graph_data
from lib.signal import lowpass_filter
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import timedelta
import gtsam
import pymap3d as pm
from tqdm.notebook import tqdm


DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


# Sensors

In [6]:
session = Session(engine)
query = select(Roll).join(RollFile).join(File).where(File.type == 'racebox').where(Roll.id.in_(
  select(Roll.id).join(RollFile).join(File).where(File.type == 'fit')
))
rolls = session.execute(query).scalars().all()
rolls

[Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
 Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
 Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
 Roll(id=37, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=1),
 Roll(id=38, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=2),
 Roll(id=39, roll_date_id=7, buggy_id=3, driver_id=3, roll_number=3),
 Roll(id=1401, roll_date_id=155, buggy_id=3, driver_id=3, roll_number=4),
 Roll(id=45, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=4)]

In [3]:
files = [{rf.file.type: rf for rf in roll.roll_files} | {'roll': roll} for roll in rolls]

In [4]:
roll_data = []
for roll in files:
  roll_data.append((roll['roll'], roll['fit'], roll['racebox'], False))
  if 'fit_c' in roll:
    roll_data.append((roll['roll'], roll['fit_c'], roll['racebox'], True))
roll_data

[(Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
  RollFile(id=89, roll_id=44, file_id=37, local_start_ms=3488, local_end_ms=262595),
  RollFile(id=3444, roll_id=44, file_id=3169, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
  RollFile(id=3255, roll_id=1388, file_id=3062, local_start_ms=3315, local_end_ms=260653),
  RollFile(id=3450, roll_id=1388, file_id=3174, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
  RollFile(id=3253, roll_id=1387, file_id=3060, local_start_ms=3457, local_end_ms=204606),
  RollFile(id=3451, roll_id=1387, file_id=3175, local_start_ms=None, local_end_ms=None),
  False),
 (Roll(id=37, roll_date_id=6, buggy_id=1, driver_id=5, roll_number=1),
  RollFile(id=3312, roll_id=37, file_id=3056, local_start_ms=None, local_end_ms=None),
  RollFile(id=3452, roll_id=37, file_id=3176, local_start_m

In [41]:
roll, fit_file, racebox_file, is_c = roll_data[0]
print(roll.id, is_c)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

fit_start_time = estimate_fit_timestamp(fit)
racebox_start_time = racebox_file.file.start_time
print(fit_start_time, racebox_start_time)

fit_record_speed = pd.DataFrame.from_records(fit['record_mesgs']).set_index('timestamp')
fit_record_speed.index *= 1000

fig = go.Figure()
fig.add_trace(go.Scatter(x=racebox_graph_data['gps_data'].index + ((racebox_start_time - fit_start_time).total_seconds()) * 1000, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox'))
fig.add_trace(go.Scatter(x=fit_gps.index, y=fit_gps.speed, mode='lines', name='fit'))
# fig.add_trace(go.Scatter(x=fit_record_speed.index, y=fit_record_speed.enhanced_speed, mode='lines', name='fit record'))
# from lib.gpx import calculate_speed
# calced_speed = calculate_speed(fit_gps)
# fig.add_trace(go.Scatter(x=calced_speed.index, y=calced_speed, mode='lines', name='calculated speed'))
fig.show()

44 False
2026-03-29 12:07:21.375000 2026-03-29 12:08:40.240000


In [7]:
px.line(racebox_graph_data['gps_data'].elevation)

0/m, 1, 2, 3m, 4*, (5m, 6*), 8/, 9*m, 10

## Load

In [24]:
from gtsam.symbol_shorthand import X, V, B, R, O
from pymap3d import geodetic2enu
from lib.geo import load_elevation_data
from scipy.interpolate import RectBivariateSpline
from pyproj import Transformer
lat0, lon0, alt0 = 40.44163016, -79.94165829, 288.42151354


In [25]:
from scipy.signal import butter, filtfilt

def estimate_racebox_offset(virb_gyro, rb_gyro, offset_ms):
    """Extra ms to add to start_time-aligned racebox timestamps, from gyro envelope xcorr."""
    def envelope(t_ms, xyz):
        fs = 1000 / np.median(np.diff(t_ms))
        mag = np.linalg.norm(xyz, axis=1)
        b, a = butter(2, 6.0 / (fs / 2), 'low')
        return t_ms / 1000, filtfilt(b, a, np.abs(mag - np.median(mag)))

    vt, venv = envelope(virb_gyro.index.to_numpy(float), virb_gyro.to_numpy(float)[:, :3])
    rt, renv = envelope(rb_gyro.timestamp.to_numpy(float) + offset_ms, rb_gyro[['x', 'y', 'z']].to_numpy())
    if min(vt[-1], rt[-1]) - max(vt[0], rt[0]) < 30:
        return 0.0
    fs = 25.0
    grid = np.arange(max(vt[0], rt[0]) + 1, min(vt[-1], rt[-1]) - 1, 1 / fs)
    b, a = butter(2, [0.3 / (fs / 2), 6.0 / (fs / 2)], 'bandpass')
    v = filtfilt(b, a, np.interp(grid, vt, venv))
    r = filtfilt(b, a, np.interp(grid, rt, renv))
    v = (v - v.mean()) / (v.std() + 1e-12)
    r = (r - r.mean()) / (r.std() + 1e-12)
    n = int(10 * fs)
    c = np.correlate(v, r, 'full')[len(r) - 1 - n: len(r) + n]
    k = int(np.argmax(c))
    frac = 0.5 * (c[k - 1] - c[k + 1]) / (c[k - 1] - 2 * c[k] + c[k + 1]) if 0 < k < len(c) - 1 else 0.0
    return float((k - n + frac) / fs * 1000)

In [26]:
def make_elevation_spline(elevation, lat0, lon0, alt0):
    data = elevation.read(1)
    nrows, ncols = data.shape

    transform = elevation.transform
    col_coords = np.array([transform.c + (c + 0.5) * transform.a for c in range(ncols)])
    row_coords = np.array([transform.f + (r + 0.5) * transform.e for r in range(nrows)])

    to_wgs84 = Transformer.from_crs(elevation.crs, "EPSG:4326", always_xy=True)

    col_mesh, row_mesh = np.meshgrid(col_coords, row_coords)
    lon_grid, lat_grid = to_wgs84.transform(col_mesh, row_mesh)

    east_grid, north_grid, _ = geodetic2enu(
        lat_grid, lon_grid, np.zeros_like(lat_grid), lat0, lon0, alt0
    )

    east_axis  = east_grid[0, :]    # east values along cols (constant row)
    north_axis = north_grid[:, 0]   # north values along rows (constant col)

    # RectBivariateSpline requires strictly increasing axes.
    if east_axis[0] > east_axis[-1]:
        east_axis = east_axis[::-1]
        data = data[:, ::-1]

    if north_axis[0] > north_axis[-1]:
        north_axis = north_axis[::-1]
        data = data[::-1, :]

    return RectBivariateSpline(east_axis, north_axis, data.T)

In [27]:
elevation_data = load_elevation_data()
elevation_spline = make_elevation_spline(elevation_data, lat0, lon0, alt0)

In [63]:
def skew_symmetric(v):
  return np.array([
    [0, -v[2], v[1]],
    [v[2], 0, -v[0]],
    [-v[1], v[0], 0]
  ], dtype=np.float64)

# Matrix to select y, z compoenents of a vector
S = np.array([[0.0, 1.0, 0.0], [0.0, 0.0, 1.0]])

In [64]:
def heading_error(this, values, jacobians):
  pose = values.atPose3(this.keys()[0])
  v_w = values.atVector(this.keys()[1])
  R_vi = values.atRot3(this.keys()[2]).matrix() # rotation from imu frame to vehicle frame
  R_wi = pose.rotation().matrix() # rotation from imu frame to world frame
  
  v_i = R_wi.T @ v_w # Velocity in imu frame
  v_v = R_vi @ v_i # Velocity in vehicle frame
  
  speed = np.linalg.norm(v_w)
  v_target = np.array([speed, 0.0, 0.0])
  
  # We want the lateral and vertical velocity to be 0
  error = v_v - v_target
  
  if jacobians is not None:
    v_i_hat = skew_symmetric(v_i)
    J_pose = np.hstack((R_vi @ v_i_hat, np.zeros((3, 3))))
    
    J_Rvi = -R_vi @ v_i_hat
    
    J_target_vw = np.zeros((3, 3))
    J_target_vw[0, :] = v_w / (speed + 1e-8)
    J_vw = (R_vi @ R_wi.T) - J_target_vw
    
    jacobians[0] = J_pose
    jacobians[1] = J_vw
    jacobians[2] = J_Rvi
  return error

heading_constraint_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5, 0.5]))

In [65]:
def elevation_error(this, values, jacobians):
    pose = values.atPose3(this.keys()[0])
    offset = values.atVector(this.keys()[1])[0]
    t = pose.translation()
    east, north, z = t[0], t[1], t[2]
    error = np.array([z - elevation_spline.ev(east, north) - offset])

    if jacobians is not None:
        dz_de = elevation_spline.ev(east, north, dx=1)
        dz_dn = elevation_spline.ev(east, north, dy=1)
        # de/dt in world frame
        de_dt = np.array([-dz_de, -dz_dn, 1.0])

        R = pose.rotation().matrix()
        jacobians[0] = np.zeros((1, 6))
        jacobians[0][0, 3:6] = de_dt @ R
        # d(error)/d(offset)
        jacobians[1] = np.array([[-1.0]])

    return error

elevation_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0]))

In [66]:
def terrain_normal(east, north):
    df_de = elevation_spline.ev(east, north, dx=1)  # d(elev)/d(east)
    df_dn = elevation_spline.ev(east, north, dy=1)  # d(elev)/d(north)
    n = np.array([-df_de, -df_dn, 1.0])
    return n / np.linalg.norm(n)


def normal_error(this, values, jacobians):
    pose = values.atPose3(this.keys()[0])
    R_vi = values.atRot3(this.keys()[1]).matrix()  # imu -> vehicle frame
    R_wi = pose.rotation().matrix()                # imu -> world frame

    t = pose.translation()
    normal = terrain_normal(t[0], t[1])

    c = R_vi.T @ np.array([0.0, 0.0, 1.0])
    up_world = R_wi @ c

    error = up_world - normal

    if jacobians is not None:
        J_rot = R_wi @ skew_symmetric(c)
        # wrt pose
        jacobians[0] = np.zeros((3, 6)) # ignores position
        jacobians[0][:, 0:3] = -J_rot
        # wrt the imu->vehicle extrinsic rotation R_vi
        jacobians[1] = J_rot

    return error

normal_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.3, 0.3, 0.3]))

In [67]:
USE_CPP_FACTORS = True

if USE_CPP_FACTORS:
  import srs_factors
  _tx, _ty = elevation_spline.get_knots()
  _kx, _ky = elevation_spline.degrees
  cpp_spline = srs_factors.Spline2D(_tx, _ty, elevation_spline.get_coeffs().reshape(len(_tx) - _kx - 1, len(_ty) - _ky - 1), _kx, _ky)
  def make_elevation_factor(i): return srs_factors.ElevationFactor(X(i), O(0), cpp_spline, elevation_noise)
  def make_normal_factor(i): return srs_factors.NormalFactor(X(i), R(0), cpp_spline, normal_noise)
  def make_heading_factor(i): return srs_factors.HeadingFactor(X(i), V(i), R(0), heading_constraint_noise)
  def make_dopp_factor(wi, dj, z): return srs_factors.BiasedVelocityFactor(wi, dj, z, doppler_noise)
  def make_lowpass_factor(wi, wp, vp, beta): return srs_factors.LowpassStateFactor(wi, wp, vp, beta, lp_proc_noise)
else:
  def make_elevation_factor(i): return gtsam.CustomFactor(elevation_noise, [X(i), O(0)], elevation_error)
  def make_normal_factor(i): return gtsam.CustomFactor(normal_noise, [X(i), R(0)], normal_error)
  def make_heading_factor(i): return gtsam.CustomFactor(heading_constraint_noise, [X(i), V(i), R(0)], heading_error)
  def _dopp_bias_error(z):
    def f(this, values, jacobians):
      if jacobians is not None:
        jacobians[0] = np.eye(3); jacobians[1] = np.eye(3)
      return values.atVector(this.keys()[0]) + values.atVector(this.keys()[1]) - z
    return f
  def make_dopp_factor(wi, dj, z): return gtsam.CustomFactor(doppler_noise, [wi, dj], _dopp_bias_error(z))
  def _lowpass_error(beta):
    def f(this, values, jacobians):
      w, wp, vp = (values.atVector(this.keys()[k]) for k in range(3))
      if jacobians is not None:
        jacobians[0] = np.eye(3); jacobians[1] = -beta * np.eye(3); jacobians[2] = -(1 - beta) * np.eye(3)
      return w - beta * wp - (1 - beta) * vp
    return f
  def make_lowpass_factor(wi, wp, vp, beta): return gtsam.CustomFactor(lp_proc_noise, [wi, wp, vp], _lowpass_error(beta))

In [69]:
idx = 9
roll, fit_file, racebox_file, is_c = roll_data[idx]
print(roll.id, is_c)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

start = {
  4: 170_000,
  3: 172_000,
  1: 57_000,
  6: 42_000,
  5: 44_000,
  9: 229_000
}[idx]
fit_gps = fit_gps.loc[start:]
fit_graphs['gps_data'] = fit_graphs['gps_data'].loc[start:]

1401 False


In [70]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
accel_cal = calibration_data['accelerometer']
accel_raw, accel, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'x': 'accel_x', 'y': 'accel_y', 'z': 'accel_z'})
accel = accel[['x', 'y', 'z']]
# probably not really necessary, but there is a bunch of non white high frequency noise
accel = pd.DataFrame(lowpass_filter(accel.T, 5, 100).T, columns=['x', 'y', 'z'], index=accel.index)

gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
gyro = gyro[['x', 'y', 'z']] * (np.pi / 180) 
# gyro = pd.DataFrame(lowpass_filter(gyro.T, 5, 100).T, columns=['x', 'y', 'z'], index=gyro.index) 

magnet_cal = calibration_data['compass']
mag_raw, magnet, mag_fs = get_sensor_data(magnet_cal, fit['magnetometer_data_mesgs'], {'x': 'mag_x', 'y': 'mag_y', 'z': 'mag_z'})
magnet = magnet[['x', 'y', 'z']]

In [71]:
g_body = np.array(accel.loc[start-1000:start].mean())
m_body = np.array(magnet.loc[start-1000:start].mean())

up = -g_body / np.linalg.norm(g_body)
east = np.cross(m_body, up)
east = east / np.linalg.norm(east)
north = np.cross(up, east)
north = north / np.linalg.norm(north)
R_wi = np.vstack((east, north, up))
accel = accel * -9.81 # gtsam expects specific force in m/s^2, and the accelerometer data is in g's 

In [72]:
fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

vel = np.array(fit_gps.velocity.to_list())
# project z direction of velocity to fit heightmap
dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy

In [73]:
from scipy.interpolate import interp1d

# 9 of 10 gps samples are dead-reckoned from doppler: keep one real fix per second as
# gps anchors, and advance the doppler stream by its per-file filter lag
def flag_stutter(t_s, e, n, v):
    dt = np.diff(t_s)
    resid = np.hypot(np.diff(e) - 0.5 * (v[:-1, 0] + v[1:, 0]) * dt,
                     np.diff(n) - 0.5 * (v[:-1, 1] + v[1:, 1]) * dt)
    mad = np.nanmedian(np.abs(resid - np.nanmedian(resid)))
    thresh = max(0.3, np.nanmedian(resid) + 6 * 1.4826 * mad)
    bad = np.zeros(len(t_s), bool)
    bad[1:] |= resid > thresh
    bad[:-1] |= resid > thresh
    return bad, thresh

def doppler_lag(t_s, e, n, v, anchor_idx):
    at = t_s[anchor_idx]
    aiv_t = (at[1:] + at[:-1]) / 2
    aiv = np.hypot(np.diff(e[anchor_idx]), np.diff(n[anchor_idx])) / np.diff(at)
    dopp = np.hypot(v[:, 0], v[:, 1])
    moving = interp1d(t_s, dopp, bounds_error=False)(aiv_t) > 3
    best = (0.0, np.inf)
    for lag in np.arange(-0.5, 2.51, 0.05):
        di = interp1d(t_s - lag, dopp, bounds_error=False)(aiv_t[moving])
        ok = ~np.isnan(di)
        if ok.sum() < 20: continue
        r = np.sqrt(np.mean((di[ok] - aiv[moving][ok]) ** 2))
        if r < best[1]: best = (lag, r)
    return best[0]

gps_t = fit_enu.index.to_numpy(float) / 1000
bad, stutter_thresh = flag_stutter(gps_t, fit_enu.x.values, fit_enu.y.values, vel)
ok_idx = np.flatnonzero(~bad)
anchor_idx = pd.Series(ok_idx, index=np.floor(gps_t[ok_idx]).astype(int)).groupby(level=0).first().values
anchors = set(anchor_idx)
dopp_lag_s = doppler_lag(gps_t, fit_enu.x.values, fit_enu.y.values, vel, anchor_idx)
# receiver filtering is modeled in-graph as a one-pole tracking state (LowpassStateFactor);
# the xcorr lag underestimates the effective delay of heavy-tail files by ~2x
DOPP_LP_T = float(np.clip(1.75 * dopp_lag_s, 0.3, 2.5))
print(f'anchors {len(anchor_idx)}/{len(gps_t)}  stutter thresh {stutter_thresh:.2f} m  doppler lag {dopp_lag_s:.2f} s  lp T {DOPP_LP_T:.2f} s')

anchors 180/1536  stutter thresh 0.80 m  doppler lag 0.50 s  lp T 0.87 s


In [74]:
# # evalute spline in grid
# bounds = (fit_enu.x.min() - 10, fit_enu.x.max() + 10, fit_enu.y.min() - 10, fit_enu.y.max() + 10)
# x = np.linspace(bounds[0], bounds[1], 100)
# y = np.linspace(bounds[2], bounds[3], 100)
# z = elevation_spline(x, y).T

# # plot as 3d surface
# fig = go.Figure(data=[go.Surface(z=z, x=x, y=y, colorscale='Viridis', opacity=0.8)])
# fig.add_trace(go.Scatter3d(x=fit_enu.x, y=fit_enu.y, z=fit_enu.z, mode='markers', name='fit GPS', marker=dict(size=2, color='red')))
# fig.update_layout(scene=dict(
#     xaxis_title='East (m)',
#     yaxis_title='North (m)',
#     zaxis_title='Elevation (m)',
#     aspectmode='manual',
#     aspectratio=dict(x=3, y=3, z=1)
# ), margin=dict(l=0, r=0, b=0, t=0))
# fig.show()

## Optimize

In [75]:
graph = gtsam.NonlinearFactorGraph()
estimate = gtsam.Values()

In [76]:
gps_base_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([3.0, 3.0, 5.0]))
doppler_base_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.3, 0.3, 1000.0]))
# gps_noise = gps_base_noise
# doppler_noise = doppler_base_noise
m_estimator = gtsam.noiseModel.mEstimator.Cauchy(2.0)
gps_noise = gtsam.noiseModel.Robust.Create(m_estimator, gps_base_noise)
doppler_noise = gtsam.noiseModel.Robust.Create(m_estimator, doppler_base_noise)

bias_rw_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([1e-3, 1e-3, 1e-3, 1e-3, 1e-3, 1e-3]) * 1e-1) # accel, gyro
imu_params = gtsam.PreintegrationParams.MakeSharedU(9.81)
imu_params.setAccelerometerCovariance(np.eye(3) * 3e-3)
imu_params.setGyroscopeCovariance(np.diag([3, 3, 1]) * 1e-3)
imu_params.setIntegrationCovariance(np.eye(3) * 1e-8)
current_bias = gtsam.imuBias.ConstantBias(g_body - (-up), np.array(gyro.mean()))
pim = gtsam.PreintegratedImuMeasurements(imu_params, current_bias)

# slowly-varying doppler bias (absorbs the receiver's low-frequency velocity drift)
DOPP_BIAS_RW = 0.05     # m/s per sqrt(s)
DOPP_BIAS_PRIOR = 0.5   # m/s
lp_proc_noise = gtsam.noiseModel.Isotropic.Sigma(3, 0.02)

In [77]:
start_pose = gtsam.Pose3(gtsam.Rot3(R_wi), gtsam.Point3(fit_enu.values[0]))
start_vel = gtsam.Point3(vel[0])
# Initial guess of roration from imu frame to vehicle frame
R_vi_start = gtsam.Rot3(np.array([
  [ 0.0, 1.0, 0.0],
  [-1.0, 0.0, 0.0],
  [ 0.0, 0.0, 1.0]
]))
if is_c:  R_vi_start = R_vi_start.compose(gtsam.Rot3.Yaw(np.deg2rad(180)))
offset_start = np.array([0.5])

graph.add(gtsam.PriorFactorPose3(X(0), start_pose, gtsam.noiseModel.Diagonal.Sigmas(np.array([5.0, 5.0, 5.0, 5.0, 5.0, 5.0]))))
graph.add(gtsam.PriorFactorPoint3(V(0), start_vel, gtsam.noiseModel.Diagonal.Sigmas(np.array([1.0, 1.0, 1.0]))))
graph.add(gtsam.PriorFactorConstantBias(B(0), current_bias, gtsam.noiseModel.Diagonal.Sigmas(np.array([1, 1, 1, 1, 1, 1]) * 1e-2)))
graph.add(gtsam.PriorFactorRot3(R(0), R_vi_start, gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5, 0.5]) * 1e3)))
graph.add(gtsam.PriorFactorVector(O(0), offset_start, gtsam.noiseModel.Diagonal.Sigmas(np.array([10.0]))))

estimate.insert(X(0), start_pose)
estimate.insert(V(0), start_vel)
estimate.insert(B(0), current_bias)
estimate.insert(R(0), R_vi_start)
estimate.insert(O(0), offset_start)

In [78]:
heading_cutoff = 0.3
rot_init = [start_pose.rotation()]
n_dbias, d_prev, t_prev_d = 0, None, 0.0
w_prev, i_prev_w = None, None
for i in range(1, fit_enu.shape[0] - 20):
  t_prev = fit_enu.index[i-1]
  t_curr = fit_enu.index[i]
  
  accel_data = accel.loc[t_prev:t_curr].values
  gyro_data = gyro.loc[t_prev:t_curr].values
  if accel_data.shape[0] == 0 or gyro_data.shape[0] == 0:
    print(f"Skipping index {i} due to missing IMU data")
    continue
  
  dt = 1.0 / 100.0
  for a, g in zip(accel_data, gyro_data):
    pim.integrateMeasurement(a, g, dt)
  
  graph.add(gtsam.ImuFactor(X(i-1), V(i-1), X(i), V(i), B(i-1), pim))
  graph.add(gtsam.BetweenFactorConstantBias(B(i-1), B(i), gtsam.imuBias.ConstantBias(), bias_rw_noise))
  if i in anchors: # interpolated samples carry no position information
    graph.add(gtsam.GPSFactor(X(i), fit_enu.values[i], gps_noise))
  if n_dbias == 0 or i in anchors: # advance the doppler bias chain once per second
    dj = gtsam.symbol('d', n_dbias)
    if n_dbias == 0:
      graph.add(gtsam.PriorFactorPoint3(dj, np.zeros(3), gtsam.noiseModel.Isotropic.Sigma(3, DOPP_BIAS_PRIOR)))
    else:
      graph.add(gtsam.BetweenFactorPoint3(d_prev, dj, np.zeros(3),
                gtsam.noiseModel.Isotropic.Sigma(3, DOPP_BIAS_RW * np.sqrt(max((fit_enu.index[i] - t_prev_d) / 1000, 0.1)))))
    estimate.insert(dj, np.zeros(3))
    d_prev, t_prev_d, n_dbias = dj, fit_enu.index[i], n_dbias + 1
  wi = gtsam.symbol('w', i)
  estimate.insert(wi, vel[i])
  if w_prev is None:
    graph.add(gtsam.PriorFactorPoint3(wi, vel[i], gtsam.noiseModel.Isotropic.Sigma(3, 1.0)))
  else:
    beta = float(np.exp(-((fit_enu.index[i] - fit_enu.index[i_prev_w]) / 1000) / DOPP_LP_T))
    graph.add(make_lowpass_factor(wi, w_prev, V(i_prev_w), beta))
  w_prev, i_prev_w = wi, i
  graph.add(make_dopp_factor(wi, d_prev, vel[i]))
  graph.add(make_elevation_factor(i))
  graph.add(make_normal_factor(i))
  prev_pose = estimate.atPose3(X(i-1))
  
  speed = np.linalg.norm(vel[i])
  if speed > heading_cutoff: # only apply heading constraint when moving
    graph.add(make_heading_factor(i))
    # use velocity for yaw and the terrain normal for pitch/roll
    yaw = np.arctan2(vel[i][1], vel[i][0]) # arctan2(east, north)
    # if is_c: yaw += np.deg2rad(180)
    
    up = terrain_normal(fit_enu.values[i-1][0], fit_enu.values[i-1][1])
    forward = np.array([np.cos(yaw), np.sin(yaw), 0.0]) # heading in the horizontal plane
    forward = forward - np.dot(forward, up) * up # project onto the terrain tangent plane
    forward = forward / np.linalg.norm(forward)
    left = np.cross(up, forward)
    R_wv_guess = gtsam.Rot3(np.column_stack([forward, left, up]))
    R_wi_guess = R_wv_guess.compose(R_vi_start)
    rot_init.append(R_wi_guess)
    prev_pose = gtsam.Pose3(R_wi_guess, fit_enu.values[i-1])
  else:
    rot_init.append(prev_pose.rotation())
    
  navstate = pim.predict(gtsam.NavState(prev_pose, vel[i-1]), current_bias)
  estimate.insert(X(i), navstate.pose())
  estimate.insert(V(i), navstate.velocity())
  estimate.insert(B(i), current_bias)
  
  pim.resetIntegrationAndSetBias(current_bias)

In [79]:
params = gtsam.LevenbergMarquardtParams()
params.setMaxIterations(100)
params.setVerbosityLM('SUMMARY')
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, estimate, params)
result = optimizer.optimize()

Initial error: 3.6e+09, values: 6242
iter      cost      cost_change    lambda  success iter_time
   0      4.9e+07      3.6e+09      1e-05      1       0.09
   1        3e+05      4.8e+07      1e-06      1       0.08
   2      4.8e+03      2.9e+05      1e-07      1       0.06
   3      3.7e+03      1.2e+03      1e-08      1       0.06
   4      3.6e+03           20      1e-09      1       0.06
   5      3.6e+03          1.3      1e-10      1       0.07
   6      3.6e+03         0.33      1e-11      1       0.07
   7      3.6e+03          0.1      1e-12      1       0.07
   8      3.6e+03        0.051      1e-13      1       0.07
   9      3.6e+03        0.019      1e-14      1       0.07


In [80]:
optimized_poses = []
optimized_vels = []
optimized_biases = []
for i in range(fit_enu.shape[0]):
  if result.exists(X(i)):
    optimized_poses.append(result.atPose3(X(i)))
    optimized_vels.append(result.atPoint3(V(i)))
    optimized_biases.append(result.atConstantBias(B(i)))
optimized_speeds = np.linalg.norm(np.array(optimized_vels)[:, :2], axis=1)

In [81]:
from gtsam import Marginals
marginals = Marginals(graph, result)
pose_marginals = []
velocity_marginals = []
speed_marginals = []
for i in range(fit_enu.shape[0]):
  if result.exists(X(i)):
    pose_marginals.append(marginals.marginalCovariance(X(i)))
    velocity_marginals.append(marginals.marginalCovariance(V(i)))
    speed_jacobian = np.zeros((1, 3))
    speed_jacobian[0, :2] = result.atPoint3(V(i))[:2] / optimized_speeds[i]
    speed_cov = speed_jacobian @ marginals.marginalCovariance(V(i)) @ speed_jacobian.T
    speed_marginals.append(speed_cov[0, 0])


## Analyze

In [82]:
racebox_enu = np.array(geodetic2enu(racebox_graph_data['gps_data'].lat, racebox_graph_data['gps_data'].long, racebox_graph_data['gps_data'].elevation, lat0, lon0, alt0)).T
racebox_enu: pd.DataFrame = pd.DataFrame(racebox_enu, index=racebox_graph_data['gps_data'].index, columns=['x', 'y', 'z'])

# align racebox to the virb timeline: start_time estimate + imu gyro-envelope lock
racebox_offset_ms = (racebox_file.file.start_time - estimate_fit_timestamp(fit)).total_seconds() * 1000
racebox_offset_ms += estimate_racebox_offset(gyro, racebox_graph_data['gyroscope'], racebox_offset_ms)
racebox_enu.index = racebox_graph_data['gps_data'].index + racebox_offset_ms

RB_CAM_OFFSETS = {}  # roll_id: (forward, left) m, camera relative to racebox; fill from measurements
fwd, left = RB_CAM_OFFSETS.get(roll.id, (0.0, 0.0))
if fwd or left:
    i0 = np.maximum(np.arange(len(racebox_enu)) - 12, 0)
    i1 = np.minimum(np.arange(len(racebox_enu)) + 12, len(racebox_enu) - 1)
    h = np.arctan2(racebox_enu.x.values[i1] - racebox_enu.x.values[i0], racebox_enu.y.values[i1] - racebox_enu.y.values[i0])
    racebox_enu.x += fwd * np.sin(h) - left * np.cos(h)
    racebox_enu.y += fwd * np.cos(h) + left * np.sin(h)

In [83]:
pose_arr = np.array([pose.translation() for pose in optimized_poses])
fig = go.Figure()
timestamps = fit_gps.index[:len(optimized_poses)]
fig.add_trace(go.Scatter(x=pose_arr[:, 0], y=pose_arr[:, 1], mode='markers', name='optimized', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=fit_enu.x, y=fit_enu.y, mode='markers', name='fit gps', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=racebox_enu.x, y=racebox_enu.y, mode='markers', name='racebox gps', hovertext=[f'timestamp: {ts}' for ts in racebox_enu.index]))
fig.show()

In [84]:
optimized_speeds = np.linalg.norm(np.array(optimized_vels)[:, :2], axis=1)
optimized_speeds = pd.Series(optimized_speeds, index=fit_gps.index[:len(optimized_vels)])
fig = go.Figure()
# plot speeds with error bands
fig.add_trace(go.Scatter(x=optimized_speeds.index, y=optimized_speeds.values, mode='lines', name='optimized speed'))
# fig.add_trace(go.Scatter(x=optimized_speeds.index, y=optimized_speeds.values, mode='lines', name='optimized speed'))

fig.add_trace(go.Scatter(x=racebox_enu.index, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox speed'))
fig.add_trace(go.Scatter(x=fit_gps.index , y=fit_gps.speed, mode='lines', name='fit gps speed'))
fig.add_trace(go.Scatter(x=list(optimized_speeds.index) + list(optimized_speeds.index)[::-1], y=list(optimized_speeds.values + 2 * np.sqrt(speed_marginals)) + list((optimized_speeds.values - 2 * np.sqrt(speed_marginals))[::-1]), fill='toself', fillcolor='rgba(0,100,80,0.2)', line=dict(color='rgba(255,255,255,0)')))
fig.show()

In [132]:
R_vi_est = result.atRot3(R(0))
yaws = [result.atPose3(X(i)).rotation().compose(R_vi_est).yaw() for i in range(len(optimized_poses))]
init_yaws = [rot.compose(R_vi_est).yaw() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': yaws, 'initial guess': init_yaws}, index=fit_gps.index[:len(optimized_poses)]))

In [133]:
# R_vi_est = result.atRot3(R(0))
yaws = [result.atPose3(X(i)).rotation().yaw() for i in range(len(optimized_poses))]
init_yaws = [rot.yaw() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': yaws, 'initial guess': init_yaws}, index=fit_gps.index[:len(optimized_poses)]))

In [134]:
pitchs = [result.atPose3(X(i)).rotation().compose(R_vi_est).pitch() for i in range(len(optimized_poses))]
init_pitchs = [rot.compose(R_vi_est).pitch() for rot in rot_init]
# px.line(pd.Series(yaws, index=fit_gps.index[:len(optimized_poses)]))
px.line(pd.DataFrame({'optimized': pitchs, 'initial guess': init_pitchs}, index=fit_gps.index[:len(optimized_poses)]))

In [137]:
vehicle_accel = []
world_accel = []
R_vi = result.atRot3(R(0)).matrix()
# R_vi = R_vi_start.matrix()
for i in range(1, fit_enu.shape[0] - 20):
  t_prev = fit_enu.index[i-1]
  t_curr = fit_enu.index[i]
  
  accel_data = accel.loc[t_prev:t_curr]
  # print(accel_data)
  gyro_data = gyro.loc[t_prev:t_curr]
  if accel_data.shape[0] == 0 or gyro_data.shape[0] == 0:
    print(f"Skipping index {i} due to missing IMU data")
    continue
  R_wi_est = result.atPose3(X(i-1)).rotation().matrix()
  bias = result.atConstantBias(B(i-1))
  accel_data = accel_data - bias.accelerometer()
  
  accel_v = accel_data.values @ R_vi.T
  accel_w = accel_data.values @ R_wi_est.T
  # accel_w = R_wi_est[:, [1] * len(accel_data)].T
  vehicle_accel.append(np.hstack([accel_data.index.values[:, None], accel_v]))
  world_accel.append(np.hstack([accel_data.index.values[:, None], accel_w]))

In [138]:
px.line(accel.loc[start:])

In [139]:
px.line(pd.DataFrame(np.vstack(vehicle_accel), columns=['timestamp', 'x', 'y', 'z']).set_index('timestamp'))

In [ ]:
px.line(pd.DataFrame(np.vstack(world_accel), columns=['timestamp', 'x', 'y', 'z']).set_index('timestamp'))

In [251]:
# calcualte energy as 1/2v^2 + gh
optimized_energies = 1/2 * np.array([optimized_vels[i].dot(optimized_vels[i]) for i in range(len(optimized_vels))])
optimized_energies += np.array([optimized_poses[i].translation()[2] for i in range(len(optimized_poses))]) * 9.81
px.line(pd.Series(optimized_energies, index=fit_gps.index[:len(optimized_poses)]), title='Optimized Energy')

In [ ]:
pos_speeds = np.linalg.norm(fit_enu.diff().iloc[1:].values, axis=1)
px.line(pd.Series(pos_speeds, index=fit_gps.index[1:]), title='Position Speed')

In [ ]:
vels_sum = vel[:, 1].cumsum()
pos = fit_enu.y
# plot velocity cumsum against position
px.line(pd.DataFrame({'position': pos, 'velocity_cumsum': vels_sum/10}, index=fit_gps.index))

In [ ]:
px.line(np.array([optimized_vels[i].dot(optimized_vels[i]) for i in range(len(optimized_vels))]))

In [ ]:
px.line(np.array([optimized_poses[i].translation()[2] for i in range(len(optimized_poses))]))

In [73]:
optimized_energies

array([2841.08304005, 2840.77492977, 2840.48777482, ..., 2900.10283214,
       2899.94256595, 2899.72222948], shape=(1772,))

In [ ]:
px.line(optimized_vels)

In [ ]:
px.line(vel)

In [ ]:
px.line(accel.loc[start:])

In [ ]:
px.line(np.array([bias.accelerometer() for bias in optimized_biases]))

In [10]:
# pan right/left, tilt up/down, roll clockwise/counterclockwise
fit = load_fit_file(f'{DATA_PATH}/archive/tmp/calibrate/2026-06-18-21-27-45.fit')

Caching archive_tmp_calibrate_2026-06-18-21-27-45.json


In [11]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'x': 'accel_x', 'y': 'accel_y', 'z': 'accel_z'})
accel_data = accel_data[['x', 'y', 'z']] * -9.81

In [ ]:
px.line(accel_data)

In [ ]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
gyro_data = gyro_data[['x', 'y', 'z']] * -(np.pi / 180) 
px.line(gyro_data)

In [ ]:
# # Vehicle should be oriented roughly in the direction of travel
# # TODO: make this aware of slip angles while turning

# def calc_heading_jacobians(R_vi, R_wi, v_i):
#   # Derivative with repect to pose
#   v_i_hat = skew_symmetric(v_i)
#   J_Rwi = S @ R_vi @ v_i_hat
  
#   J_pose = np.zeros((2, 6), dtype=np.float64)
#   J_pose[:, :3] = J_Rwi
  
#   # Derivative with repsect to world velocity
#   J_vw = S @ R_vi @ R_wi.T
  
#   # Derivative with respect to extrinsic rotation
#   J_Rvi = -S @ R_vi @ v_i_hat
  
#   return J_pose, J_vw, J_Rvi


# def heading_constraint(this, values, jacobians):
#   pose = values.atPose3(this.keys()[0])
#   v_w = values.atVector(this.keys()[1])
#   R_vi = values.atRot3(this.keys()[2]).matrix() # rotation from imu frame to vehicle frame
#   R_wi = pose.rotation().matrix() # rotation from imu frame to world frame
  
#   v_i = R_wi.T @ v_w # Velocity in imu frame
#   v_v = R_vi @ v_i # Velocity in vehicle frame
  
#   # We want the lateral and vertical velocity to be 0
#   error = S @ v_v
  
#   if jacobians is not None:
#     J_pose, J_vw, J_Rvi = calc_heading_jacobians(R_vi, R_wi, v_i)
#     jacobians[0] = J_pose
#     jacobians[1] = J_vw
#     jacobians[2] = J_Rvi
#   return error

# heading_constraint_noise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.5, 0.5]))

# Visual

## Exports

In [ ]:
roll_vid_data = []
for roll in files:
  if 'video_preview' in roll:
    roll_vid_data.append((roll['roll'], roll['fit'], roll['video_preview'], roll['racebox']))
roll_vid_data

[(Roll(id=44, roll_date_id=8, buggy_id=1, driver_id=5, roll_number=3),
  RollFile(id=89, roll_id=44, file_id=37, local_start_ms=3488, local_end_ms=262595),
  RollFile(id=88, roll_id=44, file_id=63, local_start_ms=None, local_end_ms=None),
  RollFile(id=3444, roll_id=44, file_id=3169, local_start_ms=None, local_end_ms=None)),
 (Roll(id=1388, roll_date_id=154, buggy_id=3, driver_id=3, roll_number=6),
  RollFile(id=3255, roll_id=1388, file_id=3062, local_start_ms=3315, local_end_ms=260653),
  RollFile(id=3256, roll_id=1388, file_id=3063, local_start_ms=None, local_end_ms=None),
  RollFile(id=3450, roll_id=1388, file_id=3174, local_start_ms=None, local_end_ms=None)),
 (Roll(id=1387, roll_date_id=154, buggy_id=4, driver_id=2, roll_number=7),
  RollFile(id=3253, roll_id=1387, file_id=3060, local_start_ms=3457, local_end_ms=204606),
  RollFile(id=3254, roll_id=1387, file_id=3061, local_start_ms=None, local_end_ms=None),
  RollFile(id=3451, roll_id=1387, file_id=3175, local_start_ms=None, loca

In [51]:
idx = 0
roll, fit_file, vid, racebox_file = roll_vid_data[idx]
print(roll.id)
fit = load_fit_file(resolve_path(fit_file.file.uri))
fit_gps = get_gps_data(fit)
fit_graphs = get_fit_graph_data(fit)
racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])

fit_start_time = estimate_fit_timestamp(fit)
racebox_start_time = racebox_file.file.start_time
print(fit_start_time, racebox_start_time)

fig = go.Figure()
fig.add_trace(go.Scatter(x=racebox_graph_data['gps_data'].index + ((racebox_start_time - fit_start_time).total_seconds()) * 1000, y=racebox_graph_data['gps_data'].speed, mode='lines', name='racebox'))
fig.add_trace(go.Scatter(x=fit_gps.index , y=fit_gps.speed, mode='lines', name='fit'))
from lib.gpx import calculate_speed
calced_speed = calculate_speed(fit_gps)
fig.add_trace(go.Scatter(x=calced_speed.index, y=calced_speed, mode='lines', name='calculated speed'))
fig.show()

44


2026-03-29 12:07:21.375000 2026-03-29 12:08:40.240000


In [52]:
racebox_enu = np.array(geodetic2enu(racebox_graph_data['gps_data'].lat, racebox_graph_data['gps_data'].long, racebox_graph_data['gps_data'].elevation, lat0, lon0, alt0)).T
racebox_enu: pd.DataFrame = pd.DataFrame(racebox_enu, index=racebox_graph_data['gps_data'].index, columns=['x', 'y', 'z'])
fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
# fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

vel = np.array(fit_gps.velocity.to_list())
# project z direction of velocity to fit heightmap
# dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
# dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
# vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy

In [53]:
# pose_arr = np.array([pose.translation() for pose in optimized_poses])
fig = go.Figure()
timestamps = fit_gps.index
# fig.add_trace(go.Scatter(x=pose_arr[:, 0], y=pose_arr[:, 1], mode='markers', name='optimized', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=fit_enu.x, y=fit_enu.y, mode='markers', name='fit gps', hovertext=[f'timestamp: {ts}' for ts in timestamps]))
fig.add_trace(go.Scatter(x=racebox_enu.x, y=racebox_enu.y, mode='markers', name='racebox gps', hovertext=[f'timestamp: {ts}' for ts in racebox_enu.index]))
fig.show()

In [41]:
roll_starts = {
  1: 55_000,
  3: 170_000,
  4: 41_000,
  6: 229_000,
  7: 106_000,
}
roll_start = roll_starts[idx]
vid_path = resolve_path(vid.file.uri)

partial_roll_starts = {
  0: 90_000,
  2: 20_000,
  5: 128_000,
}

## Video imu list

In [46]:
VIRB_VIDEO_START_OFFSET_MS = {'default': 130}  # ms; add per-serial measured values

def get_roll_data(idx, start):
  roll, fit_file, vid, racebox_file = roll_vid_data[idx]
  fit = load_fit_file(resolve_path(fit_file.file.uri))
  fit_gps = get_gps_data(fit)
  fit_graphs = get_fit_graph_data(fit)
  camera_start = get_camera_starts(fit)[0]
  camera_end = get_camera_ends(fit)[0]
  # first frame leads the fit video_start event by a per-camera constant (~95-160ms, n=186 study)
  serial = fit['file_id_mesgs'][0].get('serial_number')
  camera_start_correction_ms = VIRB_VIDEO_START_OFFSET_MS.get(serial, VIRB_VIDEO_START_OFFSET_MS['default'])
  camera_start -= camera_start_correction_ms
  calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
  calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
  
  accel_cal = calibration_data['accelerometer']
  accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'acc_x': 'accel_x', 'acc_y': 'accel_y', 'acc_z': 'accel_z'})
  accel_data = accel_data[['acc_x', 'acc_y', 'acc_z']] * -9.81          # g -> specific force m/s^2
  accel_data = accel_data.loc[start:camera_end]
  
  gyro_cal = calibration_data['gyroscope']
  gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'gyro_x': 'gyro_x', 'gyro_y': 'gyro_y', 'gyro_z': 'gyro_z'})
  gyro_data = gyro_data[['gyro_x', 'gyro_y', 'gyro_z']] * -(np.pi / 180)   # deg/s -> rad/s
  gyro_full = gyro_data
  gyro_data = gyro_data.loc[start:camera_end]
  imu = pd.merge_asof(accel_data, gyro_data, on='timestamp')
  imu.timestamp = (imu.timestamp * 1e6).astype('int64')
  
  if fit_gps is None:
    gps_data = pd.DataFrame({'lat': [], 'long': [], 'alt': [], 'timestamp': []}, index=[])
    velocity = pd.DataFrame({'vx': [], 'vy': [], 'vz': [], 'timestamp': []}, index=[])
  else:
    fit_enu = np.array(geodetic2enu(fit_gps.position_lat, fit_gps.position_long, fit_graphs['gps_data'].elevation, lat0, lon0, alt0)).T
    fit_enu: pd.DataFrame = pd.DataFrame(fit_enu, index=fit_gps.index, columns=['x', 'y', 'z'])
    fit_enu.z = elevation_spline.ev(fit_enu.x, fit_enu.y)

    vel = np.array(fit_gps.velocity.to_list())
    # project z direction of velocity to fit heightmap
    dx = np.array([elevation_spline.ev(x, y, dx=1) for x, y in zip(fit_enu.x, fit_enu.y)])
    dy = np.array([elevation_spline.ev(x, y, dy=1) for x, y in zip(fit_enu.x, fit_enu.y)])
    vel[:, 2] = vel[:, 0] * dx + vel[:, 1] * dy
    
    # use lat/long, but height from fit_enu
    gps_data = pd.DataFrame({'lat': fit_gps.position_lat, 'long': fit_gps.position_long, 'alt': fit_enu.z, 'timestamp': fit_gps.index * 1_000_000}, index=fit_gps.index)
    velocity = pd.DataFrame({'vx': vel[:, 0], 'vy': vel[:, 1], 'vz': vel[:, 2], 'timestamp': fit_gps.index * 1_000_000}, index=fit_gps.index)
    
  racebox_graph_data = get_racebox_graph_data(racebox_file.file.uri.split('/')[-1])
  
  fit_start_time = estimate_fit_timestamp(fit)
  racebox_start_time = racebox_file.file.start_time
  
  offset = (racebox_start_time - fit_start_time).total_seconds() * 1000
  racebox_graph_data['gps_data'].index += offset
  racebox_offset_ms = estimate_racebox_offset(gyro_full, racebox_graph_data['gyroscope'], offset)
  
  racebox_gps = pd.DataFrame({'lat': racebox_graph_data['gps_data'].lat, 'long': racebox_graph_data['gps_data'].long, 'alt': racebox_graph_data['gps_data'].elevation + 288.4, 'timestamp': racebox_graph_data['gps_data'].index * 1_000_000}, index=racebox_graph_data['gps_data'].index)
  racebox_speed = pd.DataFrame({'speed': racebox_graph_data['gps_data'].speed, 'timestamp': racebox_graph_data['gps_data'].index * 1_000_000}, index=racebox_graph_data['gps_data'].index)
  
  vid_path = vid.file.uri.replace('[[archive]]', 'archive').replace('[[videos]]', 'videos')
  data = dict(vid_path=vid_path, camera_start=camera_start * 1_000_000, racebox_offset_ms=racebox_offset_ms,
              camera_start_correction_ms=camera_start_correction_ms,
              imu_data=imu.to_dict(orient='records'), 
              gps_data=gps_data.to_dict(orient='records'), velocity=velocity.to_dict(orient='records'),
              racebox_gps=racebox_gps.to_dict(orient='records'), racebox_speed=racebox_speed.to_dict(orient='records'))
  print(data['camera_start'], imu.timestamp.min(), imu.timestamp.max(), f'racebox_offset_ms={racebox_offset_ms:+.0f}')
  return data

In [50]:
import json
for idx, start in tqdm(roll_starts.items()):
  data = get_roll_data(idx, start)
  with open(f'{DATA_PATH}/archive/vid_imu/{roll_vid_data[idx][0].id}.json', 'w') as f:
    json.dump(data, f)
for idx, start in tqdm(partial_roll_starts.items()):
  data = get_roll_data(idx, start)
  with open(f'{DATA_PATH}/archive/partial_vid_imu/{roll_vid_data[idx][0].id}.json', 'w') as f:
    json.dump(data, f)

  0%|          | 0/5 [00:00<?, ?it/s]

3315000000 55005000000 260407000000 racebox_offset_ms=+364
3180000000 170000000000 350038000000 racebox_offset_ms=+307
3208000000 41008000000 211320000000 racebox_offset_ms=+269
3572000000 229002000000 406622000000 racebox_offset_ms=+340
3555000000 106000000000 278779000000 racebox_offset_ms=+285


  0%|          | 0/3 [00:00<?, ?it/s]

3488000000 90010000000 262366000000 racebox_offset_ms=+381
3457000000 20007000000 204362000000 racebox_offset_ms=-945
3306000000 128004000000 316316000000 racebox_offset_ms=+230


In [17]:
data['gps_data']

[{'lat': 40.441783759742975,
  'long': -79.9415799882263,
  'alt': 576.7321948106935,
  'timestamp': 99231000000},
 {'lat': 40.44178074225783,
  'long': -79.94158149696887,
  'alt': 576.732353983336,
  'timestamp': 99331000000},
 {'lat': 40.44177764095366,
  'long': -79.94158267043531,
  'alt': 576.7325733721261,
  'timestamp': 99431000000},
 {'lat': 40.44177470728755,
  'long': -79.94158376008272,
  'alt': 576.7324259534914,
  'timestamp': 99531000000},
 {'lat': 40.4417719412595,
  'long': -79.94158526882529,
  'alt': 576.7311610110917,
  'timestamp': 99631000000},
 {'lat': 40.44176917523146,
  'long': -79.94158711284399,
  'alt': 576.7295845945619,
  'timestamp': 99731000000},
 {'lat': 40.44176674447954,
  'long': -79.94158912450075,
  'alt': 576.7284676965744,
  'timestamp': 99831000000},
 {'lat': 40.44176397845149,
  'long': -79.94159046560526,
  'alt': 576.7307843204763,
  'timestamp': 99931000000},
 {'lat': 40.44176146388054,
  'long': -79.94159230962396,
  'alt': 576.73415361683

## Localize smoothing

In [ ]:
<roll>.npz â 40 arrays, one row per frame (1,637 here):

pose         centre (n,3) Â· quat (n,4) Â· arc_pnp Â· lat_off Â· cov (n,6,6)
solve        ok Â· n_corr Â· n_inl Â· inl_ratio Â· rep_p50 Â· rep_p90
wrong-hill   inl_arc_span Â· inl_arc_med
support      n_sup_img Â· sup_run_mask      <- which map runs held each frame up
bootstrap    is_anchor Â· boot_ok Â· boot_n_inl Â· boot_accepted Â· arc_boot Â· arc_interpolated
intrinsics   focal Â· k1 Â· intr_prov Â· intr_seed Â· intr_seed_prov
consistency  chord_prev Â· chord_next Â· arc_loo_resid
identity     roll Â· file_id Â· video Â· frame_idx Â· t_ms Â· dec_idx Â· n_kp Â· in_map Â· sync Â· meta

## PnP smoothing

Fuse PnP localizations (`data/archive/pnp_poses/`, from the hloc PnP pipeline) with the VIRB IMU in a
GTSAM graph: states at frame timestamps, `ImuFactor` chains between them, PnP positions as unary
factors. Speed comes out of the velocity states; evaluated against racebox doppler per segment.

Findings (all eight VIRB runs: 39/44/1387 localized by PnP, plus every run that built the rtk_base
map - 37/38/45/1388/1401, whose model camera centres are exported by
`tmp/smooth_refit/export_model_centres.py`):
- the racebox<->camera shift is MEASURED per run (`SM_EVAL_DT`): scan the shift minimising the
  median distance between camera centres and lever-arm-corrected racebox positions, after removing
  the per-session racebox datum offset (0.15-2.16 m). The old hand-set values were already right
  for 39/44/1387 (within 16 ms); the five map runs had none and need +110 to +286 ms. The lever arm
  is user-measured only for 39/44/1387 - for the map runs it is an estimate, and 1 m of lever arm
  trades against ~100 ms of shift, so those taus inherit that uncertainty.
- the racebox doppler channel lags its own position channel by ~144 ms (median over the eight, but
  scatter -94..-222 ms so it is a population property, not a constant); its internal
  position-vs-doppler check alone gives 55-74 ms. Speed rmse at the position alignment therefore
  carries that lag; one common -0.144 s puts every run on the same footing:
  1388 0.115, 39 0.134, 1401 0.138, 37 0.144, 44 0.144, 38 0.149, 1387 0.178, 45 0.187 m/s.
- at the adopted alignment: 39 0.169, 44 0.210, 1387 0.181, 37 0.189, 38 0.192, 45 0.258,
  1388 0.130, 1401 0.192 m/s speed rmse, ~zero bias, positions move well under the factor sigma
  (p50 0.06-0.10 m).
- the map runs' apparent advantage over the localized PnP runs was mostly an alignment artifact -
  their old numbers (1388 0.117, 1401 0.150) happened to sit at their speed-optimal shift, and
  37/38/45 reproduce that mechanism exactly (0.158/0.161/0.180 untouched vs 0.189/0.192/0.258
  honest). With all five map runs in, the common footing interleaves map and PnP runs across
  0.115-0.149 - the gap is gone.
- run 45 is the one straggler (0.187): its own doppler lag is -222 ms rather than the shared -144,
  and its racebox POSITION channel is displaced up to ~7 m over the first ~40 s of the roll (the map
  is fine there - its camera centres sit 0.5 m from the other map runs' tracks, same as elsewhere).
  Its tau survives that (clean second half +293 vs +286 ms whole-track) but is the weakest of the
  eight along with 37's, whose position basin is the flattest (+-40 ms).
- baseline done fairly (local quadratic derivative of the raw PnP positions over +-1 s) is
  0.17-0.24 m/s: the smoother beats it by 27-44% on the six good-IMU runs on the common footing,
  by only 7% on 45, and LOSES to it on 1387 (0.178 vs 0.168).
- speed error is a repeatable function of PLACE, not of run (new E-N panel coloured by speed error,
  shared +-0.40 m/s scale): binned into 25 m bins along the course the eight profiles correlate
  pairwise at r 0.53-0.91 (mean 0.78), leave-one-out R^2 0.52-0.88, amplitude -0.23..+0.34 m/s
  (up to 2.7% of local speed) against a between-run sd of 0.04 m/s. All eight share the rtk_base
  map, so the simplest reading is a locally varying along-track scale error in it - the 40-80 m
  correlated map bias, this time seen directly rather than inferred; place-fixed doppler multipath
  is the alternative this data cannot exclude (robo RTK on the same course would separate them).
  Locally: the chute is a coherent negative-bias patch on all eight (-0.01..-0.09 m/s) whose
  |error| tracks curvature on five (sideslip - racebox reports course-over-ground - plus lateral
  map bias); the push segments alternate sign at the stride frequency (aliasing, not bias); the
  freeroll is near-white, which is why it carries no timing information.
- fit doppler makes things worse even after lag correction (receiver lowpass lags 0.6-1.3 s) - left out.
- integration dt must cover the full inter-state interval (tail after the last IMU sample included);
  truncation showed up as a speed-proportional (+2-5%) bias.
- 1387 (older VIRB): its IMU makes speed worse at any weighting (0.34 vs 0.18, re-checked against
  the corrected reference) - the linear const-velocity smoother (no_imu) wins there. On good-IMU
  runs dropping the IMU costs 8-40%. The one apparent exception, 45 preferring no_imu at the
  position alignment (0.222 vs 0.253), flips at pos+dop (0.209 vs 0.188): a mis-timed reference
  penalises the sharper IMU-informed velocity, so that is a timing symptom, not an IMU verdict.
- seq-BA-refined positions change nothing: speed error is dominated by 40-80 m correlated map
  bias (measured via residual autocorrelation + noise injection on 1388), invisible to a 12 m seq window.
- accelerometer preintegration cov inflated ~20x (acc_scale; keep gyro tight): the stock 3e-3
  over-trusts a vibrating chassis and integrates ~0.5 m spurious z arches (freeroll energy
  artifacts) that override the PnP unaries. Re-swept over all eight, 5-50 are equivalent (within
  0.005 m/s); only acc_scale=1 is clearly bad (positions detach up to 3 m).
- PnP orientation priors (ori=True; camera is the IMU device, fixed mount averaged from a
  first pass) still pay on good-IMU runs: 39 0.169 vs 0.178, 44 0.210 vs 0.222. Still hurt every
  map run (38 0.201 vs 0.191, 1388 0.144 vs 0.135).
- model positions get sigma 0.5, not 0.2, on all five map runs - the anchored map's own held-out RTK
  accuracy is ~0.36 m. racebox priors were in the mapping BA so the position agreement is partly
  circular, but BA never saw doppler so the speed comparison is fair.
- sheet: tmp/smooth_review/review.html (generator tmp/smooth_refit/gen_review.py).


In [ ]:
import json as _json
from pathlib import Path as _Path

import gtsam
import numpy as np
from gtsam.symbol_shorthand import B, V, X

SM_DATA = _Path('../../../data')
SM_TIME_OFFSETS = {'39': -50.0, '44': -50.0, '1387': -1300.0}
# measured per run, not assumed: shift minimising the median distance between the camera centres
# and the lever-arm-corrected racebox positions (tmp/smooth_refit/step1_align.py)
SM_EVAL_DT = {'39': -0.15, '44': -0.145, '1387': -0.215, '37': -0.20, '38': -0.22,
              '45': -0.285, '1388': -0.11, '1401': -0.225}
# per-run: poses file, pnp sigma, robust pnp noise, imu covariance scale,
# no_imu -> linear const-velocity smoother (cv_q = white-accel PSD)
SM_RUNS = {
    '39': dict(poses='39.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0, ori=True),
    '44': dict(poses='44.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0, ori=True),
    '1387': dict(poses='1387.json', sigma=0.5, robust=True, imu_scale=1.0,
                 no_imu=True),  # its older IMU makes speed worse; const-vel wins
    # every run that built the rtk_base map, solved model camera centres
    '37': dict(poses='model_37.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0),
    '38': dict(poses='model_38.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0),
    '45': dict(poses='model_45.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0),
    '1388': dict(poses='model_1388.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0),
    '1401': dict(poses='model_1401.json', sigma=0.5, robust=True, imu_scale=1.0, acc_scale=20.0),
}


def sm_load(run):
    cfg = SM_RUNS[run]
    vd = 'partial_vid_imu' if (SM_DATA / f'archive/partial_vid_imu/{run}.json').exists() else 'vid_imu'
    d = _json.load(open(SM_DATA / f'archive/{vd}/{run}.json'))
    imu_t = np.array([s['timestamp'] for s in d['imu_data']], float) / 1e9
    acc = np.array([[s['acc_x'], s['acc_y'], s['acc_z']] for s in d['imu_data']])
    gyr = np.array([[s['gyro_x'], s['gyro_y'], s['gyro_z']] for s in d['imu_data']])
    off = (d.get('racebox_offset_ms', 0.0) - SM_TIME_OFFSETS.get(run, 0.0)) / 1e3 \
        - SM_EVAL_DT.get(run, 0.0)
    rb_t = np.array([s['timestamp'] for s in d['racebox_gps']], float) / 1e9 + off
    rb_spd = np.array([s['speed'] for s in d['racebox_speed']], float)
    rows = _json.load(open(SM_DATA / f'archive/pnp_poses/{cfg["poses"]}'))
    rows.sort(key=lambda r: int(r['name'].split('/')[1].split('.')[0]))
    pnp_t = np.array([int(r['name'].split('/')[1].split('.')[0]) for r in rows], float) / 1e9
    pnp_p = np.array([r['center'] for r in rows])
    pnp_q = np.array([r.get('quat_cw') or [0, 0, 0, 1] for r in rows])
    return dict(cfg=cfg, imu_t=imu_t, acc=acc, gyr=gyr, rb_t=rb_t, rb_spd=rb_spd,
                pnp_t=pnp_t, pnp_p=pnp_p, pnp_q=pnp_q)

In [ ]:
def sm_solve(run):
    s = sm_load(run)
    cfg = s['cfg']
    imu_t, acc, gyr = s['imu_t'], s['acc'], s['gyr']
    pnp_t, pnp_p = s['pnp_t'], s['pnp_p']

    # gyro sign check vs track yaw rate (export convention differs from raw fit)
    grad_p = np.gradient(pnp_p[:, :2], pnp_t, axis=0)
    yr_tr = np.gradient(np.unwrap(np.arctan2(grad_p[:, 1], grad_p[:, 0])), pnp_t)
    gz = np.interp(pnp_t, imu_t, gyr[:, 2], left=np.nan, right=np.nan)
    mv = (np.hypot(*grad_p.T) > 3) & ~np.isnan(gz) & (np.abs(yr_tr) < 2)
    if mv.sum() > 30 and np.corrcoef(yr_tr[mv], gz[mv])[0, 1] < 0:
        gyr = -gyr

    # gravity: standstill window, else long low-rotation window (motion averages out)
    gmag = np.convolve(np.linalg.norm(gyr, axis=1), np.ones(200) / 200, 'same')
    spd_i = np.interp(imu_t, s['rb_t'], s['rb_spd'])
    cand = np.flatnonzero((spd_i < 0.5) & (gmag < np.percentile(gmag, 20)))
    if len(cand) > 200:
        w0, hw = int(cand[len(cand) // 2]), 100
    else:
        W = 2000
        gs = np.convolve(np.linalg.norm(gyr, axis=1), np.ones(W) / W, 'same')
        w0, hw = int(np.clip(np.argmin(gs), W // 2, len(acc) - W // 2)), W // 2
    g_body = acc[max(0, w0 - hw):w0 + hw].mean(axis=0)
    b_up = g_body / np.linalg.norm(g_body)
    b_fwd = np.array([0, 1.0, 0]) - np.dot([0, 1.0, 0], b_up) * b_up
    b_fwd /= np.linalg.norm(b_fwd)
    R_vb = np.stack([b_fwd, np.cross(b_up, b_fwd), b_up])

    st_t = pnp_t
    N = len(st_t)
    pos_i = pnp_p.copy()
    grad_s = np.gradient(pos_i[:, :2], st_t, axis=0)
    vel_i = np.stack([grad_s[:, 0], grad_s[:, 1], np.gradient(pos_i[:, 2], st_t)], 1)

    if cfg.get('no_imu'):  # linear const-velocity smoother
        P = lambda k: gtsam.symbol('p', k)
        cv_q = cfg.get('cv_q', 1.0)

        def cv_error(dt_):
            def f(this, values, jacobians):
                p1, p2, v1, v2 = (values.atPoint3(this.keys()[j]) for j in range(4))
                if jacobians is not None:
                    jacobians[0] = -np.eye(3)
                    jacobians[1] = np.eye(3)
                    jacobians[2] = -0.5 * dt_ * np.eye(3)
                    jacobians[3] = -0.5 * dt_ * np.eye(3)
                return p2 - p1 - 0.5 * dt_ * (v1 + v2)
            return f

        graph = gtsam.NonlinearFactorGraph()
        est = gtsam.Values()
        base = gtsam.noiseModel.Isotropic.Sigma(3, cfg['sigma'])
        pnp_noise = gtsam.noiseModel.Robust.Create(
            gtsam.noiseModel.mEstimator.Cauchy(2.0), base) if cfg['robust'] else base
        for k in range(N):
            est.insert(P(k), gtsam.Point3(pos_i[k]))
            est.insert(V(k), vel_i[k])
            graph.add(gtsam.PriorFactorPoint3(P(k), pnp_p[k], pnp_noise))
            if k > 0:
                dt_ = max(st_t[k] - st_t[k - 1], 1e-3)
                graph.add(gtsam.CustomFactor(
                    gtsam.noiseModel.Isotropic.Sigma(3, cv_q * dt_ ** 1.5 / np.sqrt(3)),
                    [P(k - 1), P(k), V(k - 1), V(k)], cv_error(dt_)))
                graph.add(gtsam.BetweenFactorPoint3(V(k - 1), V(k), np.zeros(3),
                          gtsam.noiseModel.Isotropic.Sigma(3, cv_q * np.sqrt(dt_))))
        params = gtsam.LevenbergMarquardtParams()
        params.setMaxIterations(50)
        result = gtsam.LevenbergMarquardtOptimizer(graph, est, params).optimize()
        pos = np.array([result.atPoint3(P(k)) for k in range(N)])
        vel = np.array([result.atPoint3(V(k)) for k in range(N)])
        return dict(run=run, t=st_t, pos=pos, vel=vel,
                    spd=np.linalg.norm(vel[:, :2], axis=1), rb_t=s['rb_t'],
                    rb_spd=s['rb_spd'], pnp_p=pnp_p, graph=graph, result=result)

    def rot_at(k):
        v = grad_s[k]
        if np.linalg.norm(v) < 0.5:
            nz = np.flatnonzero(np.linalg.norm(grad_s, axis=1) > 0.5)
            v = grad_s[nz[np.abs(nz - k).argmin()]] if len(nz) else np.array([1.0, 0])
        yaw = np.arctan2(v[1], v[0])
        fwd = np.array([np.cos(yaw), np.sin(yaw), 0.0])
        R_wv = np.column_stack([fwd, np.cross([0, 0, 1.0], fwd), [0, 0, 1.0]])
        return gtsam.Rot3(R_wv @ R_vb)

    graph = gtsam.NonlinearFactorGraph()
    est = gtsam.Values()
    base = gtsam.noiseModel.Isotropic.Sigma(3, cfg['sigma'])
    pnp_noise = gtsam.noiseModel.Robust.Create(
        gtsam.noiseModel.mEstimator.Cauchy(2.0), base) if cfg['robust'] else base
    ip = gtsam.PreintegrationParams.MakeSharedU(9.81)
    ip.setAccelerometerCovariance(np.eye(3) * 3e-3 * cfg['imu_scale']
                              * cfg.get('acc_scale', 1.0))
    ip.setGyroscopeCovariance(np.diag([3, 3, 1]) * 1e-3 * cfg['imu_scale'])
    ip.setIntegrationCovariance(np.eye(3) * 1e-8)
    bias0 = gtsam.imuBias.ConstantBias(g_body - 9.81 * b_up,
                                       gyr[max(0, w0 - hw):w0 + hw].mean(axis=0))
    pim = gtsam.PreintegratedImuMeasurements(ip, bias0)
    bias_rw = gtsam.noiseModel.Diagonal.Sigmas(np.full(6, 1e-4))
    graph.add(gtsam.PriorFactorPoint3(V(0), vel_i[0], gtsam.noiseModel.Isotropic.Sigma(3, 1.0)))
    graph.add(gtsam.PriorFactorConstantBias(B(0), bias0,
              gtsam.noiseModel.Diagonal.Sigmas(np.array([.1, .1, .1, .01, .01, .01]))))
    for k in range(N):
        est.insert(X(k), gtsam.Pose3(rot_at(k), gtsam.Point3(pos_i[k])))
        est.insert(V(k), vel_i[k])
        est.insert(B(k), bias0)
        if k > 0:
            dt_ = st_t[k] - st_t[k - 1]
            m_ = np.flatnonzero((imu_t > st_t[k - 1]) & (imu_t <= st_t[k]))
            if len(m_) and dt_ < 5.0:
                dts = np.diff(imu_t[m_], prepend=st_t[k - 1])
                for j, dtj in zip(m_, dts):
                    pim.integrateMeasurement(acc[j], gyr[j], max(dtj, 1e-4))
                tail = st_t[k] - imu_t[m_[-1]]  # must cover full interval (else +v bias)
                if tail > 1e-6:
                    jn = min(m_[-1] + 1, len(acc) - 1)
                    pim.integrateMeasurement(acc[jn], gyr[jn], tail)
                graph.add(gtsam.ImuFactor(X(k - 1), V(k - 1), X(k), V(k), B(k - 1), pim))
                pim.resetIntegrationAndSetBias(bias0)
            else:
                graph.add(gtsam.BetweenFactorPoint3(V(k - 1), V(k), np.zeros(3),
                          gtsam.noiseModel.Isotropic.Sigma(3, 2.0 * np.sqrt(max(dt_, 0.1)))))
                graph.add(gtsam.BetweenFactorPose3(
                    X(k - 1), X(k),
                    gtsam.Pose3(rot_at(k - 1).inverse().compose(rot_at(k)), gtsam.Point3(0, 0, 0)),
                    gtsam.noiseModel.Diagonal.Sigmas(
                        np.concatenate([np.full(3, 1.0), np.full(3, 30.0)]))))
            graph.add(gtsam.BetweenFactorConstantBias(B(k - 1), B(k),
                      gtsam.imuBias.ConstantBias(), bias_rw))
        graph.add(gtsam.GPSFactor(X(k), pnp_p[k], pnp_noise))
    params = gtsam.LevenbergMarquardtParams()
    params.setMaxIterations(100)
    result = gtsam.LevenbergMarquardtOptimizer(graph, est, params).optimize()
    if cfg.get('ori'):
        # anchor orientations with PnP rotations via an averaged camera<-body mount
        from scipy.spatial.transform import Rotation as SR
        R_wc = SR.from_quat(s['pnp_q']).inv().as_matrix()
        Ms = [R_wc[k].T @ result.atPose3(X(k)).rotation().matrix() for k in range(N)]
        U, _, Vt = np.linalg.svd(np.sum(Ms, axis=0))
        R_cb = U @ np.diag([1, 1, np.linalg.det(U @ Vt)]) @ Vt
        spread = np.degrees([np.linalg.norm(SR.from_matrix(M @ R_cb.T).as_rotvec()) for M in Ms])
        ori_noise = gtsam.noiseModel.Isotropic.Sigma(3, np.radians(max(np.median(spread), 0.5)))
        for k in range(N):
            graph.add(gtsam.PoseRotationPrior3D(X(k), gtsam.Rot3(R_wc[k] @ R_cb), ori_noise))
        result = gtsam.LevenbergMarquardtOptimizer(graph, result, params).optimize()
    pos = np.array([result.atPose3(X(k)).translation() for k in range(N)])
    vel = np.array([result.atPoint3(V(k)) for k in range(N)])
    return dict(run=run, t=st_t, pos=pos, vel=vel, spd=np.linalg.norm(vel[:, :2], axis=1),
                rb_t=s['rb_t'], rb_spd=s['rb_spd'], pnp_p=pnp_p, graph=graph, result=result)

In [ ]:
from db.database import Roll as _Roll

sm_events = {}
for _rid in [39, 44, 1387, 37, 38, 45, 1388, 1401]:
    _r = session.get(_Roll, _rid)
    sm_events[str(_rid)] = {(f'{e.type}_{e.tag}' if e.type == 'hill_start' else e.type):
                            e.timestamp_ms for e in _r.roll_events}


def sm_eval(sol):
    t, spd = sol['t'], sol['spd']
    rb = np.interp(t, sol['rb_t'], sol['rb_spd'])
    peak = int(np.argmax(rb))
    still = np.flatnonzero(rb[:peak] < 0.5)
    k0 = int(still[-1]) if len(still) else 0
    em = np.arange(len(t)) >= k0
    serr = spd - rb
    dpos = np.linalg.norm(sol['pos'] - sol['pnp_p'], axis=1)
    ev = sm_events[sol['run']]
    ev_off = t[k0] - ev['roll_start'] / 1e3
    rows = {'all': em}
    for a, b, name in [('hill_start_1', 'freeroll_start', 'push_front'),
                       ('freeroll_start', 'chute_start', 'freeroll'),
                       ('chute_start', 'hill_start_3', 'chute'),
                       ('hill_start_3', 'roll_end', 'push_back')]:
        rows[name] = em & (t >= ev[a] / 1e3 + ev_off) & (t < ev[b] / 1e3 + ev_off)
    print(f"{sol['run']:5}: pos_delta p50 {np.median(dpos):.2f} max {dpos.max():.2f} | " +
          ' | '.join(f'{n} {np.sqrt((serr[m] ** 2).mean()):.3f}'
                     for n, m in rows.items() if m.sum() > 5))
    return rb, em


sm_sols = {run: sm_solve(run) for run in SM_RUNS}
for _run, _sol in sm_sols.items():
    sm_eval(_sol)

In [ ]:
_run = '44'
_sol = sm_sols[_run]
_rb, _em = sm_eval(_sol)
fig = go.Figure()
fig.add_trace(go.Scatter(x=_sol['t'][_em], y=_sol['spd'][_em], mode='lines', name='smoothed'))
fig.add_trace(go.Scatter(x=_sol['t'][_em], y=_rb[_em], mode='lines', name='racebox doppler'))
_pt = _sol['t']
_ps = np.linalg.norm(np.gradient(_sol['pnp_p'][:, :2], _pt, axis=0), axis=1)
fig.add_trace(go.Scatter(x=_pt[_em], y=_ps[_em], mode='markers', name='pnp finite diff',
                         marker=dict(size=3, opacity=0.4)))
fig.update_layout(title=f'run {_run} speed', xaxis_title='t (s)', yaxis_title='m/s')
fig.show()